# Notebook 02 — Baseline: Test-case Runner
**Nhóm 67 | Tuần 2**

Chạy solution mẫu MBPP qua hai bộ test (public và hidden). Tính toán các metric chính.

> Chạy notebook 00 và 01 và 02 trước.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys, json, csv
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Project")
sys.path.insert(0, str(BASE / "src"))
from runner import grade_submission, compute_fpr

DATA_FILE = BASE / "data" / "processed" / "mbpp_clean.json"
OUT_DIR = BASE / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(DATA_FILE, encoding="utf-8") as f:
    problems = json.load(f)

print(f"✓ Đọc dữ liệu: {len(problems)} bài")
print("✓ Runner sẵn sàng")
print()
print("Baseline = Test-case runner (không phải mô hình ML):")
print("  - Chạy file .py qua subprocess + timeout 5s")
print("  - Ghi nhận Pass/Fail từng test")
print("  - Phân loại lỗi: SE / WA / RE / TLE")
print("  - Tính Test Pass Rate và FPR")

✓ Đọc dữ liệu: 15 bài
✓ Runner sẵn sàng

Baseline = Test-case runner (không phải mô hình ML):
  - Chạy file .py qua subprocess + timeout 5s
  - Ghi nhận Pass/Fail từng test
  - Phân loại lỗi: SE / WA / RE / TLE
  - Tính Test Pass Rate và FPR


In [3]:
print('=' * 65)
print('  CHẠY BASELINE — SOLUTION MẪU MBPP')
print('=' * 65)

summary_rows = []
all_results  = []

for prob in problems:
    tid       = prob['task_id']
    code      = prob['code']
    topic     = prob['topic']
    pub_tests = prob['public_tests']
    hid_tests = prob['hidden_tests']

    func_name = [
        line.split('(')[0].replace('def ', '').strip()
        for line in code.split('\n')
        if line.strip().startswith('def ')
    ][0]

    pub_r = grade_submission(code, func_name, pub_tests, 'public')
    hid_r = grade_submission(code, func_name, hid_tests, 'hidden')
    fpr   = compute_fpr(pub_r, hid_r)

    fp_flag = ' ← FP' if fpr['is_false_positive'] else ''
    print(f'[{tid:02d}] {func_name}() | {topic}')
    print(f'     Public : {pub_r["pass_count"]}/{pub_r["total_count"]} ({pub_r["test_pass_rate"]}%)')
    print(f'     Hidden : {hid_r["pass_count"]}/{hid_r["total_count"]} ({hid_r["test_pass_rate"]}%) {fp_flag}')
    print()

    summary_rows.append({
        'task_id':           tid,
        'func_name':         func_name,
        'topic':             topic,
        'pub_pass':          pub_r['pass_count'],
        'pub_total':         pub_r['total_count'],
        'pub_tpr':           pub_r['test_pass_rate'],
        'pub_WA':            pub_r['error_counts']['WA'],
        'pub_RE':            pub_r['error_counts']['RE'],
        'pub_TLE':           pub_r['error_counts']['TLE'],
        'pub_SE':            pub_r['error_counts']['SE'],
        'hid_pass':          hid_r['pass_count'],
        'hid_total':         hid_r['total_count'],
        'hid_tpr':           hid_r['test_pass_rate'],
        'hid_WA':            hid_r['error_counts']['WA'],
        'hid_RE':            hid_r['error_counts']['RE'],
        'hid_TLE':           hid_r['error_counts']['TLE'],
        'hid_SE':            hid_r['error_counts']['SE'],
        'is_false_positive': fpr['is_false_positive'],
        'avg_latency_s':     hid_r['avg_latency'],
    })
    all_results.append({'task_id': tid, 'func': func_name,
                        'public': pub_r, 'hidden': hid_r, 'fpr': fpr})

  CHẠY BASELINE — SOLUTION MẪU MBPP
[01] sum_list() | list
     Public : 3/3 (100.0%)
     Hidden : 6/6 (100.0%) 

[02] is_even() | math
     Public : 3/3 (100.0%)
     Hidden : 6/6 (100.0%) 

[03] reverse_string() | string
     Public : 3/3 (100.0%)
     Hidden : 6/6 (100.0%) 

[04] find_max() | list
     Public : 3/3 (100.0%)
     Hidden : 6/6 (100.0%) 

[05] count_char() | string
     Public : 3/3 (100.0%)
     Hidden : 6/6 (100.0%) 

[06] factorial() | math
     Public : 3/3 (100.0%)
     Hidden : 6/6 (100.0%) 

[07] is_palindrome() | string
     Public : 3/3 (100.0%)
     Hidden : 6/6 (100.0%) 

[08] remove_duplicates() | list
     Public : 3/3 (100.0%)
     Hidden : 6/6 (100.0%) 

[09] average() | math
     Public : 3/3 (100.0%)
     Hidden : 6/6 (100.0%) 

[10] to_uppercase() | string
     Public : 3/3 (100.0%)
     Hidden : 6/6 (100.0%) 

[11] is_prime() | math
     Public : 3/3 (100.0%)
     Hidden : 6/6 (100.0%) 

[12] sum_even() | list
     Public : 3/3 (100.0%)
     Hidden 

In [4]:
# Tổng hợp metric
n   = len(summary_rows)
fp  = sum(1 for r in summary_rows if r["is_false_positive"])

avg_pub = round(sum(r["pub_tpr"] for r in summary_rows) / n, 2)
avg_hid = round(sum(r["hid_tpr"] for r in summary_rows) / n, 2)
avg_lat = round(sum(r["avg_latency_s"] for r in summary_rows) / n, 4)
fpr_pct = round(fp / n * 100, 2)

metric = {
    "so_bai": n,
    "public_test_per_bai": 3,
    "hidden_test_per_bai": 6,
    "avg_TPR_public_%": avg_pub,
    "avg_TPR_hidden_%": avg_hid,
    "false_positive_count": fp,
    "FPR_%": fpr_pct,
    "avg_latency_s": avg_lat,
    "error_public": {
        "SE": sum(r["pub_SE"] for r in summary_rows),
        "WA": sum(r["pub_WA"] for r in summary_rows),
        "RE": sum(r["pub_RE"] for r in summary_rows),
        "TLE": sum(r["pub_TLE"] for r in summary_rows),
    },
    "error_hidden": {
        "SE": sum(r["hid_SE"] for r in summary_rows),
        "WA": sum(r["hid_WA"] for r in summary_rows),
        "RE": sum(r["hid_RE"] for r in summary_rows),
        "TLE": sum(r["hid_TLE"] for r in summary_rows),
    },
}

print("=" * 55)
print("  BẢNG METRIC TỔNG HỢP — BASELINE")
print("=" * 55)
print(f"  Số bài:                  {n}")
print(f"  Test Pass Rate (public): {avg_pub}%")
print(f"  Test Pass Rate (hidden): {avg_hid}%")
print(f"  False Positive Rate:     {fpr_pct}% ({fp}/{n} bài)")
print(f"  Latency trung bình:      {avg_lat}s/test")
print(f"  Lỗi public: {metric['error_public']}")
print(f"  Lỗi hidden: {metric['error_hidden']}")
print("=" * 55)

# Lưu kết quả
with open(OUT_DIR / "baseline_summary.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
    w.writeheader()
    w.writerows(summary_rows)

with open(OUT_DIR / "metric_summary.json", "w", encoding="utf-8") as f:
    json.dump(metric, f, ensure_ascii=False, indent=2)

print(f"\n✓ Lưu: {OUT_DIR / 'baseline_summary.csv'}")
print(f"✓ Lưu: {OUT_DIR / 'metric_summary.json'}")

  BẢNG METRIC TỔNG HỢP — BASELINE
  Số bài:                  15
  Test Pass Rate (public): 100.0%
  Test Pass Rate (hidden): 100.0%
  False Positive Rate:     0.0% (0/15 bài)
  Latency trung bình:      0.0413s/test
  Lỗi public: {'SE': 0, 'WA': 0, 'RE': 0, 'TLE': 0}
  Lỗi hidden: {'SE': 0, 'WA': 0, 'RE': 0, 'TLE': 0}

✓ Lưu: /content/drive/MyDrive/Project/results/baseline_summary.csv
✓ Lưu: /content/drive/MyDrive/Project/results/metric_summary.json
